In [28]:
import json
import os
import re
import subprocess
from pathlib import Path
from typing import Dict, Iterable, Optional

import soundfile as sf
from datasets import Audio, load_dataset_builder

In [29]:
OUTPUT_DIR = "./ua_ast_data"

# LibriSpeech full train split is huge (~60 GB). For local runs, start with the
# 100h clean split; remove this env var when you intentionally want the full set.
os.environ["ESB_LIBRISPEECH_TRAIN_SPLITS"] = "clean.100"

# For quick debug set something like 100
MAX_SAMPLES_PER_SPLIT = None

# True -> export local wav files
# False -> keep paths from HF cache
EXPORT_AUDIO = True

# Optional custom HF cache dir
HF_CACHE_DIR = "./hf_cache"
LOCAL_ARCHIVE_DIR = "./local_archives"
os.environ["ESB_LOCAL_ARCHIVE_DIR"] = LOCAL_ARCHIVE_DIR

In [30]:
_TED_BRACE_TAG_RE = re.compile(r"\{[^}]+\}")      # {NOISE}, {COUGH}, ...
_TED_ANGLE_TAG_RE = re.compile(r"<[^>]+>")        # <sil>, ...
_TED_PAREN_NUM_RE = re.compile(r"\(\d+\)")        # for(2)
_MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_librispeech_text(text: str) -> str:
    text = text.strip().lower()
    text = _MULTI_SPACE_RE.sub(" ", text)
    return text


def normalize_tedlium_text(text: str) -> str:
    text = text.strip().lower()
    text = _TED_BRACE_TAG_RE.sub(" ", text)
    text = _TED_ANGLE_TAG_RE.sub(" ", text)
    text = _TED_PAREN_NUM_RE.sub("", text)
    text = _MULTI_SPACE_RE.sub(" ", text).strip()
    return text


def normalize_text(dataset_name: str, text: str) -> str:
    if dataset_name == "librispeech":
        return normalize_librispeech_text(text)
    if dataset_name == "tedlium":
        return normalize_tedlium_text(text)
    return text.strip().lower()

In [31]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def write_jsonl(records: Iterable[Dict], out_path: Path) -> None:
    ensure_dir(out_path.parent)
    with out_path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def export_audio_if_needed(
    audio: Dict,
    out_wav_path: Path,
    export_audio: bool,
) -> str:
    if not export_audio:
        # path from HF cache
        return audio["path"]

    ensure_dir(out_wav_path.parent)
    sf.write(out_wav_path, audio["array"], audio["sampling_rate"])
    return str(out_wav_path)

In [32]:
LIBRISPEECH_ARCHIVES = {
    "train-clean-100.tar.gz": "https://www.openslr.org/resources/12/train-clean-100.tar.gz",
    "dev-clean.tar.gz": "https://www.openslr.org/resources/12/dev-clean.tar.gz",
    "dev-other.tar.gz": "https://www.openslr.org/resources/12/dev-other.tar.gz",
    "test-clean.tar.gz": "https://www.openslr.org/resources/12/test-clean.tar.gz",
    "test-other.tar.gz": "https://www.openslr.org/resources/12/test-other.tar.gz",
}


def download_file_with_resume(url: str, out_path: Path) -> None:
    ensure_dir(out_path.parent)
    cmd = [
        "curl",
        "-L",
        "-C",
        "-",
        "--retry",
        "20",
        "--retry-delay",
        "10",
        "--connect-timeout",
        "60",
        "--speed-time",
        "300",
        "--speed-limit",
        "1024",
        "-o",
        str(out_path),
        url,
    ]
    print(f"Downloading/resuming: {out_path.name}")
    subprocess.run(cmd, check=True)


def predownload_librispeech_archives(local_archive_dir: str = LOCAL_ARCHIVE_DIR) -> None:
    archive_dir = Path(local_archive_dir)
    for filename, url in LIBRISPEECH_ARCHIVES.items():
        download_file_with_resume(url, archive_dir / filename)


In [33]:
def build_dataset(dataset_name: str, cache_dir: Optional[str] = None):
    import os
    import importlib
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
    os.environ["HF_HUB_ETAG_TIMEOUT"] = "500"
    os.environ["ESB_LOCAL_ARCHIVE_DIR"] = str(Path(LOCAL_ARCHIVE_DIR).resolve())
    import esb_datasets
    importlib.reload(esb_datasets)
    import aiohttp
    builder = esb_datasets.ESBDatasets(
        config_name=dataset_name,
        cache_dir=cache_dir,
    )

    print(f"Preparing dataset builder for: {dataset_name}")
    print(f"Using local archive dir: {os.environ.get('ESB_LOCAL_ARCHIVE_DIR')}")
    storage_options = {
        "client_kwargs": {
            "timeout": aiohttp.ClientTimeout(total=None, sock_connect=120, sock_read=None)
        }
    }
    builder.download_and_prepare(storage_options=storage_options)

    ds_dict = builder.as_dataset()
    return ds_dict

In [34]:
def build_combined_manifests(output_root: Path) -> None:
    combined_dir = output_root / "manifests" / "combined"
    ensure_dir(combined_dir)

    for split_name in ["train", "validation", "test"]:
        combined_records = []

        for dataset_name in ["librispeech", "tedlium"]:
            path = output_root / "manifests" / dataset_name / f"{split_name}.jsonl"
            if not path.exists():
                continue

            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    combined_records.append(json.loads(line))

        out_path = combined_dir / f"{split_name}.jsonl"
        write_jsonl(combined_records, out_path)
        print(f"Saved combined manifest: {out_path} ({len(combined_records)} samples)")

In [35]:
def process_split(
    ds,
    dataset_name: str,
    split_name: str,
    output_root: Path,
    max_samples: Optional[int],
    export_audio: bool,
) -> None:
    print(f"\n=== Processing {dataset_name} / {split_name} ===")

    ds = ds.cast_column("audio", Audio(sampling_rate=16000))

    if max_samples is not None:
        max_samples = max(0, min(max_samples, len(ds)))
        ds = ds.select(range(max_samples))

    manifest_records = []

    for idx, sample in enumerate(ds):
        sample_id = str(sample["id"])
        text = normalize_text(dataset_name, sample["text"])
        audio = sample["audio"]

        if not text:
            continue

        wav_path = (
            output_root
            / "audio"
            / dataset_name
            / split_name
            / f"{sample_id}.wav"
        )

        audio_path = export_audio_if_needed(
            audio=audio,
            out_wav_path=wav_path,
            export_audio=export_audio,
        )

        record = {
            "id": sample_id,
            "dataset": dataset_name,
            "split": split_name,
            "audio_path": audio_path,
            "sampling_rate": int(audio["sampling_rate"]),
            "duration_sec": round(len(audio["array"]) / float(audio["sampling_rate"]), 4),
            "source_lang": "en",
            "target_lang": "uk",
            "source_text": text,
            "target_text_uk": "",
        }
        manifest_records.append(record)

        if (idx + 1) % 500 == 0:
            print(f"Processed {idx + 1} samples...")

    manifest_path = output_root / "manifests" / dataset_name / f"{split_name}.jsonl"
    write_jsonl(manifest_records, manifest_path)

    print(f"Saved manifest: {manifest_path}")
    print(f"Total records: {len(manifest_records)}")

In [36]:
def process_dataset_dict(
    ds_dict,
    dataset_name: str,
    output_root: Path,
    max_samples: Optional[int],
    export_audio: bool,
) -> None:
    for split_name in ds_dict.keys():
        process_split(
            ds=ds_dict[split_name],
            dataset_name=dataset_name,
            split_name=split_name,
            output_root=output_root,
            max_samples=max_samples,
            export_audio=export_audio,
        )

In [37]:
def build_combined_manifests(output_root: Path) -> None:
    combined_dir = output_root / "manifests" / "combined"
    ensure_dir(combined_dir)

    split_names = set()

    for dataset_name in ["librispeech", "tedlium"]:
        dataset_manifest_dir = output_root / "manifests" / dataset_name
        if dataset_manifest_dir.exists():
            for path in dataset_manifest_dir.glob("*.jsonl"):
                split_names.add(path.stem)

    for split_name in sorted(split_names):
        combined_records = []

        for dataset_name in ["librispeech", "tedlium"]:
            path = output_root / "manifests" / dataset_name / f"{split_name}.jsonl"
            if not path.exists():
                continue

            with path.open("r", encoding="utf-8") as f:
                for line in f:
                    combined_records.append(json.loads(line))

        out_path = combined_dir / f"{split_name}.jsonl"
        write_jsonl(combined_records, out_path)
        print(f"Saved combined manifest: {out_path} ({len(combined_records)} samples)")

In [ ]:
output_root = Path(OUTPUT_DIR)
ensure_dir(output_root)

all_dataset_splits = {}

# LibriSpeech is already built locally from ./local_archives by build_librispeech_local.py.
# Do not call build_dataset("librispeech") here: the HF/fsspec downloader times out.
librispeech_manifest_dir = output_root / "manifests" / "librispeech"
if librispeech_manifest_dir.exists():
    all_dataset_splits["librispeech"] = sorted(path.stem for path in librispeech_manifest_dir.glob("*.jsonl"))
    print(f"Using existing LibriSpeech manifests: {all_dataset_splits['librispeech']}")
else:
    raise FileNotFoundError(
        "Missing LibriSpeech manifests. Run: python3 build_librispeech_local.py"
    )

# Optional: enable this when you want to build TEDLIUM too.
BUILD_TEDLIUM = False

if BUILD_TEDLIUM:
    dataset_name = "tedlium"
    print(f"\n========== {dataset_name.upper()} ==========")
    ds_dict = build_dataset(dataset_name, cache_dir=HF_CACHE_DIR)
    print(f"Available splits for {dataset_name}: {list(ds_dict.keys())}")
    all_dataset_splits[dataset_name] = list(ds_dict.keys())
    process_dataset_dict(
        ds_dict=ds_dict,
        dataset_name=dataset_name,
        output_root=output_root,
        max_samples=MAX_SAMPLES_PER_SPLIT,
        export_audio=EXPORT_AUDIO,
    )

build_combined_manifests(output_root)

print("\nDone.")
